# Notebook 1B: Image Validation

This notebook validates the images collected in 1A_DataEngineering and removes any that cannot be processed downstream. It reads `products_with_images.csv` and outputs a clean version of the same file with invalid images removed.

Validation checks:
1. File exists on disk
2. PIL can open the file (catches corrupted, truncated, or misidentified formats)
3. Image can be converted to RGB (catches unusual channel configurations)

# Setup

In [2]:
import glob
import os

from PIL import Image, ImageFile
import pandas as pd

ImageFile.LOAD_TRUNCATED_IMAGES = True  # recover partially-downloaded images

In [3]:
BASE  = os.path.abspath('../data')
IMGS  = os.path.join(BASE, 'img', 'original')
CLEAN = os.path.join(BASE, 'img', 'original_clean')
DATA  = os.path.join(BASE, 'processed')

# Load Data

In [15]:
df = pd.read_csv(f'{DATA}/products_with_images.csv')
print(f'Images in dataset: {len(df)}')

Images in dataset: 10643


## Removing Duplicates

I remove rows with duplicate `img_name` values, keeping the first occurrence. Duplicate filenames mean the same image would be processed and counted twice downstream.

In [16]:
n_before = len(df)
df = df.drop_duplicates(subset='img_name', keep='first').reset_index(drop=True)
print(f'Duplicates removed: {n_before - len(df)}')
print(f'Remaining: {len(df)}')

Duplicates removed: 1132
Remaining: 9511


# Repairing Files

## Truncated Images
Setting `ImageFile.LOAD_TRUNCATED_IMAGES = True` above allows PIL to read partially-downloaded or truncated files that would otherwise raise an `OSError`. These images may have minor artifacts at the edges but are generally usable for color extraction.

## `.web` Files
Some images were downloaded with a `.web` extension — likely WebP images or other formats saved with the wrong extension. PIL detects image format by magic bytes rather than file extension, so these files can often be opened and identified correctly. Where possible, I rename them on disk to the correct extension and update the dataframe, rather than discarding them.

In [17]:
FORMAT_TO_EXT = {
    'JPEG': '.jpg', 'PNG': '.png', 'WEBP': '.webp',
    'GIF': '.gif', 'BMP': '.bmp', 'TIFF': '.tiff',
}

repaired, unreadable = 0, 0
web_idx = df.index[df['img_name'].str.endswith('.web', na=False)].tolist()

for i in web_idx:
    old_name = df.img_name[i]
    path = os.path.join(IMGS, old_name)
    try:
        im = Image.open(path)
        ext = FORMAT_TO_EXT.get(im.format)
        if ext is None:
            unreadable += 1
            continue
        new_name = os.path.splitext(old_name)[0] + ext
        os.rename(path, os.path.join(IMGS, new_name))
        df.at[i, 'img_name'] = new_name
        repaired += 1
    except Exception:
        unreadable += 1

print(f'.web files found:    {len(web_idx)}')
print(f'Repaired (renamed):  {repaired}')
print(f'Unreadable (dropped by validation): {unreadable}')

.web files found:    104
Repaired (renamed):  104
Unreadable (dropped by validation): 0


# Image Validation

For each image in the dataset, I run four checks:
1. File exists on disk
2. File extension is a recognized image format (flags `.web` and other unsupported formats)
3. Alpha channel mean is above threshold — catches mostly-transparent images
4. Pixel standard deviation is above threshold — catches near-uniform/blank images (all white, all black)

**On check 3:** When manually annotating a stratified sample of 222 images in notebook 2B, approximately 6% (13 images) could not be annotated because they were transparent or blank — the files opened without error but contained no visible color content. This check catches those cases automatically before they reach the modeling stage.

Checks 3 and 4 use tunable thresholds defined as constants.

In [18]:
import numpy as np

VALID_EXTENSIONS    = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.tif'}
BLANK_ALPHA_THRESHOLD = 20  # mean alpha below this → mostly transparent (0-255 scale)
BLANK_STD_THRESHOLD   = 8   # pixel std dev below this → near-uniform/blank (0-255 scale)

def validate_image(img_name, imgs_dir):
    path = os.path.join(imgs_dir, img_name)
    if not os.path.exists(path):
        return 'missing: file not on disk'
    ext = os.path.splitext(img_name)[1].lower()
    if ext not in VALID_EXTENSIONS:
        return f'unsupported format: {ext} extension'
    try:
        im = Image.open(path)
    except Exception as e:
        return type(e).__name__ + ': ' + str(e)
    if im.mode in ('RGBA', 'LA', 'PA'):
        alpha = np.array(im.getchannel('A'))
        if alpha.mean() < BLANK_ALPHA_THRESHOLD:
            return f'blank: mostly transparent (mean alpha={alpha.mean():.1f})'
    try:
        rgb = np.array(im.convert('RGB'))
        if rgb.std() < BLANK_STD_THRESHOLD:
            return f'blank: near-uniform color (std={rgb.std():.1f})'
    except Exception as e:
        return type(e).__name__ + ': ' + str(e)
    return None

In [19]:
df['validation_error'] = df['img_name'].apply(lambda n: validate_image(n, IMGS))

valid = df['validation_error'].isna()
print(f'Valid images:   {valid.sum()}')
print(f'Invalid images: {(~valid).sum()}')

/Users/connie/dev/lipstick_color_extraction/.venv/lib/python3.9/site-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Valid images:   9167
Invalid images: 344


## Summary of Invalid Images

In [20]:
invalid_df = df[~valid][['img_name', 'validation_error']]
error_counts = invalid_df['validation_error'].str.split(':').str[0].value_counts().reset_index()
error_counts.columns = ['error_type', 'count']
print(error_counts.to_string(index=False))
print()
print(invalid_df.to_string(index=False))

            error_type  count
                 blank    222
UnidentifiedImageError     60
    unsupported format     53
               missing      9

                                                                                                img_name                                                                                                                                                                                    validation_error
                                                                                                     404                                                                                                                                                                           missing: file not on disk
                                        lipstick__anastasia_beverly_hills__liquid_lipstick__starfish.jpe                                                                                                                                               

# Flagging Unusable Images

Images that pass format and content checks but are too small to support clustering are flagged here. A file under 1 KB is almost certainly a placeholder or HTTP error page. An image smaller than 10×10 pixels doesn't have enough pixel diversity.

These are kept separate from the main validation checks since the threshold is a practical modeling floor, not a file quality issue.

That said, there are no images that meet these criteria.

In [21]:
MIN_FILE_SIZE_BYTES = 1024   # 1 KB
MIN_DIMENSION_PX   = 10     # width and height

for i in df.index:
    if pd.notna(df.at[i, 'validation_error']):
        continue  # already flagged
    path = os.path.join(IMGS, df.img_name[i])
    if os.path.getsize(path) < MIN_FILE_SIZE_BYTES:
        df.at[i, 'validation_error'] = f'unusable: file size < {MIN_FILE_SIZE_BYTES} bytes'
        continue
    try:
        w, h = Image.open(path).size
        if w < MIN_DIMENSION_PX or h < MIN_DIMENSION_PX:
            df.at[i, 'validation_error'] = f'unusable: dimensions too small ({w}x{h})'
    except Exception:
        pass  # already caught by validation

unusable = df['validation_error'].str.startswith('unusable', na=False)
print(f'Unusable images flagged: {unusable.sum()}')
print(df.loc[unusable, 'validation_error'].value_counts().to_string())

Unusable images flagged: 0
Series([], )


# Remove Invalid Images and Save

I remove all images that failed validation and save the clean dataset. Downstream notebooks read `products_with_images.csv`, so the output overwrites the same file.

In [22]:
n_before = len(df)
df_clean = df[valid].drop(columns=['validation_error']).reset_index(drop=True)
print(f'Removed:   {n_before - len(df_clean)} images')
print(f'Remaining: {len(df_clean)} images')

Removed:   344 images
Remaining: 9167 images


In [24]:
df_clean.to_csv(f'{DATA}/products_with_images.csv', index=False)

# Normalize Image Formats

Non-JPG formats (`.webp`, `.jpe`, `.tif`, `.gif`) do not render reliably in browser-based tools such as Label Studio. I convert every valid image to `.jpg` in `data/img/original_clean/`, leaving the originals in `data/img/original/` untouched.

This folder is the single source of images for all downstream notebooks and annotation work. It is fully reproducible — delete it and re-run this cell to regenerate.

In [5]:
df_clean = pd.read_csv(f'{DATA}/products_with_images.csv')

os.makedirs(CLEAN, exist_ok=True)

converted, skipped, failed = 0, 0, 0
failed_list = []

for img_name in df_clean.img_name:
    src  = os.path.join(IMGS, img_name)
    stem = os.path.splitext(img_name)[0]
    dst  = os.path.join(CLEAN, stem + '.jpg')

    if os.path.exists(dst):
        skipped += 1
        continue
    try:
        Image.open(src).convert('RGB').save(dst, 'JPEG')
        converted += 1
    except Exception as e:
        failed_list.append((img_name, str(e)))
        failed += 1

print(f'Converted: {converted}')
print(f'Skipped (already exists): {skipped}')
print(f'Failed:    {failed}')
if failed_list:
    for name, err in failed_list[:10]:
        print(f'  {name}: {err}')

/Users/connie/dev/lipstick_color_extraction/.venv/lib/python3.9/site-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Converted: 9167
Skipped (already exists): 0
Failed:    0
